# ComfyUI AI Influencer on Kaggle

This notebook sets up ComfyUI for AI influencer generation on Kaggle with T4 x2 GPUs.

### Instructions:
1. Enable T4 x2 GPU.
2. Turn on Internet Access.
3. Run cells sequentially.

In [ ]:
!git clone https://github.com/Dhruwkd/comfyui-kaggle-influencer.git
%cd comfyui-kaggle-influencer
!chmod +x scripts/*.sh
!./scripts/install_nodes.sh
!./scripts/download_models.sh

In [ ]:
# Install cloudflared binary to a local path (avoid "Text file busy" on /usr/local/bin)
!mkdir -p /kaggle/working/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/bin/cloudflared
!chmod +x /kaggle/working/bin/cloudflared

import subprocess
import threading
import time
import os
import sys

# Verify ComfyUI exists
comfyui_path = "/kaggle/working/ComfyUI"
if not os.path.exists(comfyui_path):
    raise RuntimeError(f"ComfyUI not found at {comfyui_path}. Did cell 1 complete successfully?")

tunnel_url = {"value": None}
tunnel_ready = threading.Event()

def run_tunnel():
    # --no-autoupdate prevents cloudflared from trying to self-update (can hang)
    proc = subprocess.Popen(
        ["/kaggle/working/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        line = line.strip()
        print(f"[cloudflared] {line}", flush=True)
        # Extract tunnel URL
        if "trycloudflare.com" in line and not tunnel_url["value"]:
            # URL format: https://xxx.trycloudflare.com
            import re
            match = re.search(r'(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)', line)
            if match:
                tunnel_url["value"] = match.group(1)
                print(f"\n{'='*60}", flush=True)
                print(f"🌐 TUNNEL URL: {tunnel_url['value']}", flush=True)
                print(f"{'='*60}\n", flush=True)
                tunnel_ready.set()

tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
tunnel_thread.start()

# Wait for tunnel URL (max 30 seconds)
print("⏳ Waiting for cloudflared tunnel to establish...", flush=True)
tunnel_ready.wait(timeout=30)

if not tunnel_url["value"]:
    print("⚠️ Tunnel URL not detected yet. Check cloudflared logs above.", flush=True)
else:
    print(f"✅ Tunnel ready: {tunnel_url['value']}", flush=True)

# Give ComfyUI a moment to be ready
time.sleep(3)

# Start ComfyUI from the correct directory
os.chdir(comfyui_path)
print(f"🚀 Starting ComfyUI from {comfyui_path} on port 8188...", flush=True)
print(f"📱 ACCESS COMPYUI AT: {tunnel_url['value'] or 'https://<your-tunnel>.trycloudflare.com'}", flush=True)
print("⚠️  DO NOT use http://localhost:8188 or http://127.0.0.1:8188 — those only work ON the Kaggle server!", flush=True)
!python main.py --lowvram --port 8188 --listen 0.0.0.0